# LAPA Hidden States Extraction

This notebook sets up the LAPA environment and extracts hidden states from the model.

**Important:** This notebook will restart the runtime after installing condacolab. You'll need to run the cells in sequence after the restart.

## Step 1: Install CondaColab (Runtime will restart)

In [ ]:
# 1) Install condacolab (one time per fresh runtime)
!pip -q install -U condacolab
import condacolab
condacolab.install()   # ⚠️ This will restart the runtime

## Step 2: Create and Activate Conda Environment

**Run this cell after the runtime restarts**

In [ ]:
# Create conda environment
!conda create -n lapaTemp python=3.10 -y

# Note: In Colab, we need to use conda run instead of activate
print("Conda environment 'lapaTemp' created successfully!")

## Step 3: Setup Project Directory and Install Dependencies

In [ ]:
# Create base thesis directory
!mkdir -p /content/thesis_temp
!mkdir -p /content/thesis/raw_datasets

# Clone the thesis repository
%cd /content/thesis_temp
!git clone https://github.com/808kalli/thesis.git

# Copy the LAPA source code to the expected location
!cp -r /content/thesis_temp/thesis/src/lapa /content/thesis/src/

# Change to the lapa directory
%cd /content/thesis/src/lapa

print("Repository cloned and LAPA source code set up successfully!")
print("Current directory:", !pwd)

## Step 4: Install Python Dependencies

In [ ]:
# Install requirements from the cloned repository
!conda run -n lapaTemp pip install -r requirements.txt

# Install JAX with CUDA support
!conda run -n lapaTemp pip install --upgrade jax==0.4.23 jaxlib==0.4.23+cuda12.cudnn89 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

# Fix orbax-checkpoint version conflict
!conda run -n lapaTemp pip install orbax-checkpoint==0.5.10

# Reinstall JAX/JAXlib
!conda run -n lapaTemp pip install --upgrade jax==0.4.23 jaxlib==0.4.23+cuda12.cudnn89 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

# Install PyTorch
!conda run -n lapaTemp pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu118

# Install CUDA dependencies
!conda run -n lapaTemp pip install "nvidia-cudnn-cu11==8.9.5.30"

# Uninstall and reinstall JAX with correct CUDA version
!conda run -n lapaTemp pip uninstall -y jax jaxlib
!conda run -n lapaTemp pip install jaxlib==0.4.23+cuda11.cudnn86 jax==0.4.23 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

print("Dependencies installed successfully!")

## Step 5: Install Hugging Face CLI and Login

In [ ]:
# Install Hugging Face CLI
!conda run -n lapaTemp pip install huggingface_hub

# Login to Hugging Face (you'll need to enter your token)
!conda run -n lapaTemp huggingface-cli login

print("Hugging Face CLI installed. Please complete the login process above.")

## Step 6: Download Dataset from Hugging Face

In [ ]:
# Download the LIBERO spatial dataset
!conda run -n lapaTemp huggingface-cli download \
  aopolin-lv/libero_spatial_no_noops_lerobot_v21 \
  --repo-type dataset \
  --include "*" \
  --local-dir /content/thesis/raw_datasets/libero_spatial \
  --local-dir-use-symlinks False

print("Dataset downloaded successfully!")

## Step 7: Download LAPA Model Weights

In [ ]:
# Create checkpoint directory
!mkdir -p /content/thesis/src/lapa/lapa_checkpoints

# Change to checkpoint directory
%cd /content/thesis/src/lapa/lapa_checkpoints

# Download model weights
!wget https://huggingface.co/latent-action-pretraining/LAPA-7B-openx/resolve/main/tokenizer.model
!wget https://huggingface.co/latent-action-pretraining/LAPA-7B-openx/resolve/main/vqgan
!wget https://huggingface.co/latent-action-pretraining/LAPA-7B-openx/resolve/main/params

# Rename params file to match your script
!mv params params_sthv2

print("Model weights downloaded successfully!")

# List downloaded files
!ls -la /content/thesis/src/lapa/lapa_checkpoints/

## Step 8: Verify Installation and Environment

In [ ]:
# Verify JAX installation and CUDA availability
!conda run -n lapaTemp python -c "
import jax
print('JAX version:', jax.__version__)
print('JAX devices:', jax.devices())
print('CUDA available:', len(jax.devices('gpu')) > 0)

import torch
print('PyTorch version:', torch.__version__)
print('PyTorch CUDA available:', torch.cuda.is_available())
"

# Check if all required files exist
print("\nChecking file structure:")
!ls -la /content/thesis/raw_datasets/libero_spatial/ | head -10
!ls -la /content/thesis/src/lapa/lapa_checkpoints/

## Step 9: Run LAPA Hidden States Extraction

The script is now ready to run since the repository has been cloned automatically.

In [ ]:
# Change to the lapa directory
%cd /content/thesis/src/lapa

# Run the LAPA hidden states extraction script
!conda run -n lapaTemp python latent_pretraining/extract_lapa_hidden_states.py \
    --dataset_dir /content/thesis/raw_datasets/libero_spatial \
    --output_dir /content/thesis \
    --vqgan_checkpoint lapa_checkpoints/vqgan \
    --load_checkpoint params::lapa_checkpoints/params_sthv2 \
    --num_episodes 1 \
    --seed 7

print("LAPA hidden states extraction completed!")

## Step 10: Check Output and Download Results

In [ ]:
# Check the output directory for generated files
print("Output directory contents:")
!ls -la /content/thesis/

# If you want to download specific output files, use:
# from google.colab import files
# files.download('/content/thesis/your_output_file.pkl')

print("\nScript execution completed! Check the output directory for your results.")

## Important Notes

1. **Runtime Restart**: The notebook will restart after installing condacolab. Continue with Step 2 after the restart.

2. **Repository Cloning**: The notebook automatically clones your thesis repository from GitHub, so no manual upload is needed.

3. **Hugging Face Login**: You'll need to provide your Hugging Face token when prompted in Step 5.

4. **GPU Memory**: Make sure you have a GPU runtime enabled in Colab for optimal performance.

5. **File Paths**: All paths have been adjusted for the Colab environment (`/content/` instead of `/workspace/`).

6. **Requirements**: The notebook will install dependencies from your requirements.txt file if it exists in the repository.